
# Reinforcement Learning in Deep Neural Networks  
## Value-Based (Q-Learning) vs Policy-Based (Policy Gradient)

This notebook explains clearly:

- Q-value (Value-based RL)
- Policy-based learning (REINFORCE)
- The "toy car retracing backward" intuition
- All symbols and equations explained
- Fully executable PyTorch examples

---

# 1. Markov Decision Process (MDP)

An RL problem is:

$$
(\mathcal{S}, \mathcal{A}, P, R, \gamma)
$$

Where:

- $\mathcal{S}$ = states  
- $\mathcal{A}$ = actions  
- $P(s'|s,a)$ = transition probability  
- $R(s,a)$ = reward  
- $\gamma \in [0,1)$ = discount factor  

Return:

$$
G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k}
$$

Objective:

$$
\max_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[G_0]
$$

---

# 2. Value-Based RL (Q-Learning)

$$
Q^\pi(s,a) = \mathbb{E}[G_t | s_t=s, a_t=a]
$$

Bellman optimality:

$$
Q^*(s,a) = \mathbb{E}[r + \gamma \max_{a'} Q^*(s',a')]
$$

Deep Q-Learning target:

$$
y_t = r_t + \gamma \max_{a'} Q_{\theta^-}(s_{t+1}, a')
$$

Loss:

$$
\mathcal{L}(\theta) = (Q_\theta(s_t,a_t) - y_t)^2
$$

---

# 3. Policy Gradient

Policy:

$$
\pi_\theta(a|s)
$$

Policy Gradient Theorem:

$$
\nabla_\theta J(\theta)
= \mathbb{E}[\nabla_\theta \log \pi_\theta(a_t|s_t) G_t]
$$

Loss:

$$
\mathcal{L} = - \log \pi_\theta(a_t|s_t) \cdot G_t
$$

If the toy car crashes, final return is negative.
All earlier actions get multiplied by that negative return.
Thus probabilities decrease — this is the backward retracing intuition.


In [ ]:

import numpy as np

class LineWorld:
    def __init__(self, goal=5, cliff=-5, max_steps=20):
        self.goal = goal
        self.cliff = cliff
        self.max_steps = max_steps
        self.reset()

    def reset(self):
        self.pos = 0
        self.steps = 0
        return np.array([self.pos], dtype=np.float32)

    def step(self, action):
        move = -1 if action == 0 else 1
        self.pos += move
        self.steps += 1

        if self.pos >= self.goal:
            return np.array([self.pos], dtype=np.float32), 10.0, True
        if self.pos <= self.cliff:
            return np.array([self.pos], dtype=np.float32), -10.0, True

        if self.steps >= self.max_steps:
            return np.array([self.pos], dtype=np.float32), -1.0, True

        return np.array([self.pos], dtype=np.float32), -1.0, False

env = LineWorld()


In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
import random

class QNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )
    def forward(self, x):
        return self.net(x)

q = QNet()
target_q = QNet()
target_q.load_state_dict(q.state_dict())

optimizer = optim.Adam(q.parameters(), lr=0.01)
gamma = 0.99

def train_q_learning(episodes=300):
    for ep in range(episodes):
        state = env.reset()
        done = False

        while not done:
            state_t = torch.tensor(state)

            if random.random() < 0.1:
                action = random.randint(0,1)
            else:
                with torch.no_grad():
                    action = torch.argmax(q(state_t)).item()

            next_state, reward, done = env.step(action)

            with torch.no_grad():
                max_next = torch.max(target_q(torch.tensor(next_state)))
                target = reward + gamma * max_next * (0 if done else 1)

            prediction = q(state_t)[action]
            loss = (prediction - target) ** 2

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            state = next_state

        if ep % 50 == 0:
            target_q.load_state_dict(q.state_dict())
            print("Q Episode:", ep)

train_q_learning()


In [ ]:

class PolicyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        logits = self.net(x)
        return torch.softmax(logits, dim=-1)

policy = PolicyNet()
optimizer_pg = optim.Adam(policy.parameters(), lr=0.01)

def train_policy_gradient(episodes=300):
    for ep in range(episodes):
        log_probs = []
        rewards = []

        state = env.reset()
        done = False

        while not done:
            state_t = torch.tensor(state)
            probs = policy(state_t)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()

            next_state, reward, done = env.step(action.item())

            log_probs.append(dist.log_prob(action))
            rewards.append(reward)

            state = next_state

        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)

        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        loss = 0
        for log_prob, G in zip(log_probs, returns):
            loss += -log_prob * G

        optimizer_pg.zero_grad()
        loss.backward()
        optimizer_pg.step()

        if ep % 50 == 0:
            print("PG Episode:", ep)

train_policy_gradient()
